<a href="https://colab.research.google.com/github/EstebanBotero03/Senalesysistemas/blob/main/PUNTO_2.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

## PUNTO 2
ESTEBAN BOTERO OROZCO


### Celda 1: Instalación de las Librerías Necesarias

Esta celda de código utiliza el comando `!pip install` para instalar todas las librerías de Python que son esenciales para que la aplicación funcione. La opción `-q` (quiet) suprime los mensajes de instalación para mantener el entorno de trabajo limpio.

* **`streamlit`**: El framework sobre el cual se construye la aplicación web interactiva.
* **`numpy`**: Es fundamental para los cálculos numéricos y la manipulación de las señales.
* **`scipy`**: Se emplea para el diseño de filtros digitales (`signal`), el cálculo de la Transformada de Fourier (`fft`) y para leer/escribir archivos `.wav` (`io.wavfile`).
* **`matplotlib`**: Genera las gráficas estáticas de las señales y sus espectros.
* **`pandas`**: Es una dependencia común para el análisis de datos y la visualización.
* **`pydub`**: Permite la conversión de archivos de audio en formato `.mp3` al formato `.wav` para su procesamiento.
* **`yt-dlp`**: Provee la funcionalidad para descargar audio directamente desde un enlace de YouTube.
* **`pyngrok`**: Crea un túnel público para que la aplicación de Streamlit, ejecutándose en el servidor de Colab, sea accesible desde un navegador web.









In [4]:
!pip install streamlit numpy scipy matplotlib pandas pydub yt-dlp pyngrok -q


### Celda 2: El Corazón de la Simulación (`streamlit_app.py`) ✍️

Esta celda utiliza el "comando mágico" de Colab (`%%writefile`) para escribir todo su contenido en un archivo llamado `streamlit_app.py`. Este archivo contiene la lógica completa de la aplicación de Streamlit.

#### **Estructura del Código:**

1.  **Importaciones y Configuración**: Se importan las librerías requeridas y se configura el título y el ícono que aparecerán en la pestaña del navegador.

2.  **Funciones de Ayuda**: Se definen varias funciones para encapsular tareas repetitivas y mejorar la legibilidad del código:
    * `plot_signal()`: Grafica una señal en el dominio del tiempo.
    * `plot_spectrum()`: Calcula la FFT y grafica el espectro de frecuencia de una señal.
    * `plot_filter()`: Visualiza la respuesta a la frecuencia de los filtros a través de un Diagrama de Bode y un diagrama de Polos y Ceros.
    * `download_yt_audio()`: Gestiona la descarga y conversión del audio proveniente de YouTube.

3.  **Interfaz y Lógica Principal**:
    * **Sidebar**: En la barra lateral (`st.sidebar`), se le presentan al usuario las opciones para configurar la simulación. Allí, el usuario selecciona el tipo de señal de entrada (pulso, YouTube o archivo local), la frecuencia de la portadora (`fc`) y el tipo de banda lateral (USB o LSB).
    * **Proceso de Modulación**:
        * **DSB-SC**: La señal mensaje es multiplicada por la portadora.
        * **Filtro SSB**: Se diseña un filtro pasa-banda Butterworth para aislar la banda lateral deseada (superior o inferior).
        * **Señal SSB**: La señal DSB-SC es procesada por el filtro para obtener la señal SSB final.
    * **Proceso de Demodulación**:
        * **Demodulación Coherente**: La señal SSB recibida se multiplica por una portadora local idéntica a la original.
        * **Filtro Pasa-Bajas (LPF)**: Se diseña un filtro para recuperar la señal de mensaje original, eliminando los componentes de alta frecuencia.
        * **Señal Recuperada**: Se aplica el LPF. La señal resultante es comparada con la original y se habilita un reproductor de audio para su escucha.




In [5]:
%%writefile streamlit_app.py
import streamlit as st
import numpy as np
import scipy.signal as signal
from scipy.fft import fft, fftfreq, fftshift
import matplotlib.pyplot as plt
from scipy.io.wavfile import read, write
from pydub import AudioSegment
import io
import os
import subprocess

# --- Page Configuration ---
st.set_page_config(
    page_title="Simulador de Modulación SSB-AM",
    page_icon="📡",
    layout="wide",
)
plt.style.use('seaborn-v0_8-darkgrid')

# --- Helper Functions (plotting, etc.) ---

def plot_signal(t, sig, title, xlabel="Tiempo (s)", y_max=0):
    fig, ax = plt.subplots(figsize=(10, 3))
    ax.plot(t, sig)
    ax.set_title(title, fontsize=14)
    ax.set_xlabel(xlabel)
    ax.set_ylabel("Amplitud")
    if y_max > 0:
        ax.set_ylim([-y_max, y_max])
    ax.grid(True)
    st.pyplot(fig)

def plot_spectrum(sig, fs, title, x_lim=None, show_negative_freq=False):
    n = len(sig)
    yf = fft(sig)
    xf = fftfreq(n, 1 / fs)

    if show_negative_freq:
        yf = fftshift(yf)
        xf = fftshift(xf)

    fig, ax = plt.subplots(figsize=(10, 4))
    ax.plot(xf, 2.0/n * np.abs(yf))
    ax.set_title(title, fontsize=14)
    ax.set_xlabel("Frecuencia (Hz)")
    ax.set_ylabel("Magnitud")

    if x_lim is not None:
        if isinstance(x_lim, (list, tuple)):
            ax.set_xlim(x_lim)
        else:
            if show_negative_freq:
                ax.set_xlim([-x_lim, x_lim])
            else:
                ax.set_xlim([0, x_lim])

    ax.grid(True)
    st.pyplot(fig)


def plot_filter(b, a, fs, title):
    w, h = signal.freqz(b, a, worN=8000)
    z, p, k = signal.tf2zpk(b, a)
    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 4))
    fig.suptitle(title, fontsize=16)
    ax1.plot(0.5 * fs * w / np.pi, 20 * np.log10(abs(h)), 'b')
    ax1.set_ylabel('Amplitud [dB]', color='b')
    ax1.set_xlabel('Frecuencia [Hz]')
    ax1.set_title("Diagrama de Bode (Magnitud)")
    ax1.grid(True)
    circulo_unitario = plt.Circle((0,0), 1, fill=False, color='gray', ls='--')
    ax2.add_artist(circulo_unitario)
    ax2.plot(np.real(z), np.imag(z), 'o', markersize=8, fillstyle='none', label='Ceros')
    ax2.plot(np.real(p), np.imag(p), 'x', markersize=8, label='Polos')
    ax2.set_aspect('equal')
    ax2.set_xlim([-1.5, 1.5])
    ax2.set_ylim([-1.5, 1.5])
    ax2.set_title("Plano de Polos y Ceros")
    ax2.set_xlabel("Eje Real")
    ax2.set_ylabel("Eje Imaginario")
    ax2.grid(True)
    ax2.legend()
    st.pyplot(fig)

def download_yt_audio(url):
    audio_filename = "downloaded_audio.wav"
    command = ['yt-dlp', '-x', '--audio-format', 'wav', '-o', 'temp_audio.%(ext)s', url]
    try:
        st.info("Descargando audio de YouTube... por favor espere.")
        subprocess.run(command, check=True, capture_output=True, text=True)
        if os.path.exists("temp_audio.wav"):
            os.rename("temp_audio.wav", audio_filename)
            st.success("Audio descargado y convertido a .wav")
            return audio_filename
    except subprocess.CalledProcessError as e:
        st.error(f"Error al descargar de YouTube: {e.stderr}")
        return None
    return None

st.title("📡 Simulador de Modulación y Demodulación SSB-AM")
st.markdown("Este dashboard visualiza el proceso de modulación SSB y su demodulación coherente.")

st.sidebar.title("Configuración")
msg_type = st.sidebar.radio(
    "1. Seleccione la fuente de la señal mensaje:",
    ('Pulso Rectangular', 'Enlace de YouTube', 'Subir Archivo de Audio')
)

fs = 44100
t = np.linspace(0, 5, 5 * fs, endpoint=False)
message = None
yt_url = None

if msg_type == 'Pulso Rectangular':
    message = np.zeros_like(t)
    message[int(0.1*fs):int(0.3*fs)] = 1
elif msg_type == 'Enlace de YouTube':
    yt_url = st.sidebar.text_input("Pegue el enlace del video de YouTube aquí:")
    if yt_url:
        audio_file = download_yt_audio(yt_url)
        if audio_file:
            fs, audio_data = read(audio_file)
            if audio_data.ndim > 1: audio_data = audio_data.mean(axis=1)
            message = audio_data.astype(float) / np.max(np.abs(audio_data))
            st.sidebar.audio(audio_file)
elif msg_type == 'Subir Archivo de Audio':
    uploaded_file = st.sidebar.file_uploader("Suba un archivo .wav o .mp3", type=["wav", "mp3"])
    if uploaded_file:
        try:
            if uploaded_file.name.endswith('.mp3'):
                audio = AudioSegment.from_mp3(uploaded_file)
                uploaded_file = io.BytesIO()
                audio.export(uploaded_file, format="wav")
            fs, audio_data = read(uploaded_file)
            if audio_data.ndim > 1: audio_data = audio_data.mean(axis=1)
            message = audio_data.astype(float) / np.max(np.abs(audio_data))
            st.sidebar.audio(uploaded_file)
        except Exception as e:
            st.sidebar.error(f"Error: {e}")

if message is not None:
    if len(message) > 5 * fs:
        message = message[:5*fs]
    t = np.linspace(0, len(message)/fs, len(message), endpoint=False)

    fc = st.sidebar.slider("2. Frecuencia de la portadora (fc) [Hz]", 1000, 10000, 5000, 100)
    sideband_type = st.sidebar.radio("3. Tipo de Banda Lateral:", ('Banda Lateral Superior (USB)', 'Banda Lateral Inferior (LSB)'))

    message_bw = 3500

    if fc <= message_bw and sideband_type == 'Banda Lateral Inferior (LSB)':
        st.error(f"Error de Diseño de Filtro: La frecuencia de la portadora ({fc} Hz) debe ser mayor que el ancho de banda del mensaje ({message_bw} Hz) para la modulación LSB.")
        st.warning("Por favor, aumente la frecuencia de la portadora en el deslizador.")
    else:
        st.header("Análisis de la Señal")
        with st.expander("Etapa 1: Señal Mensaje Original", expanded=True):
            plot_signal(t, message, "Señal Mensaje en el Tiempo")

            if msg_type == 'Pulso Rectangular':
                plot_spectrum(message, fs, "Espectro de la Señal Mensaje", x_lim=50, show_negative_freq=True)
            else:
                plot_spectrum(message, fs, "Espectro de la Señal Mensaje", x_lim=4000)

        with st.expander("Etapa 2: Modulación DSB-SC"):
            carrier = np.cos(2 * np.pi * fc * t)
            dsb_signal = message * carrier
            plot_signal(t, dsb_signal, "Señal DSB-SC", y_max=1)
            plot_spectrum(dsb_signal, fs, "Espectro DSB-SC", x_lim=(fc - 4000, fc + 4000))

        with st.expander("Etapa 3: Diseño del Filtro SSB"):
            if sideband_type == 'Banda Lateral Superior (USB)':
                cutoff_freqs = [fc, fc + message_bw]
            else: # LSB
                cutoff_freqs = [fc - message_bw, fc]
            nyquist = 0.5 * fs
            b_ssb, a_ssb = signal.butter(8, [max(1, cutoff_freqs[0])/nyquist, cutoff_freqs[1]/nyquist], btype='band')
            plot_filter(b_ssb, a_ssb, fs, f"Filtro SSB para {sideband_type}")

        with st.expander("Etapa 4: Señal Modulada SSB"):
            ssb_signal = signal.lfilter(b_ssb, a_ssb, dsb_signal)
            plot_signal(t, ssb_signal, "Señal SSB en el Tiempo", y_max=0.5)
            plot_spectrum(ssb_signal, fs, "Espectro de la Señal SSB", x_lim=(fc - 4000, fc + 4000))

        st.header("Proceso de Demodulación")
        with st.expander("Etapa 5: Demodulación Coherente"):
            demod_mult = ssb_signal * carrier
            plot_spectrum(demod_mult, fs, "Espectro post-multiplicación", x_lim=2*fc + 4000)

        with st.expander("Etapa 6: Filtro Pasa-Bajas (LPF)"):
            lpf_cutoff = message_bw
            b_lpf, a_lpf = signal.butter(6, lpf_cutoff/nyquist, btype='low')
            plot_filter(b_lpf, a_lpf, fs, "Filtro Pasa-Bajas de Recuperación")

        with st.expander("Etapa 7: Señal Mensaje Recuperada", expanded=True):
            recovered_signal = signal.lfilter(b_lpf, a_lpf, demod_mult) * 4
            fig, ax = plt.subplots(figsize=(10, 4))
            ax.plot(t, message, label="Original", alpha=0.7)
            ax.plot(t, recovered_signal, label="Recuperada", linestyle='--')
            ax.set_title("Comparación: Original vs. Recuperada")
            ax.legend()
            st.pyplot(fig)

            st.write("Escucha la señal recuperada:")
            wav_buffer = io.BytesIO()
            norm_recovered = recovered_signal / np.max(np.abs(recovered_signal)) if np.max(np.abs(recovered_signal)) > 0 else recovered_signal
            write(wav_buffer, fs, (norm_recovered * 32767).astype(np.int16))
            st.audio(wav_buffer, format='audio/wav')
else:
    st.info("Seleccione una fuente de señal en el panel izquierdo para comenzar.")

Overwriting streamlit_app.py


### Celda 3: Lanzamiento de la Aplicación 🚀

La celda final se encarga de ejecutar la aplicación y de hacerla accesible públicamente a través de internet.

1.  **Autenticación de Ngrok**:
    * En la variable `NGROK_AUTHTOKEN`, el usuario debe pegar su token de autenticación personal, el cual se obtiene desde el panel de control de [ngrok](https://dashboard.ngrok.com/get-started/your-authtoken). Este paso es indispensable para establecer la conexión.

2.  **Ejecución de Streamlit en Segundo Plano**:
    * El comando `!nohup streamlit run streamlit_app.py --server.port 8501 &` inicia la aplicación en el puerto `8501`. La ejecuta en segundo plano (`&`), lo que asegura que el proceso no se interrumpa.

3.  **Creación del Túnel Público**:
    * La línea `ngrok.connect(8501)` instruye a `ngrok` para que cree un túnel seguro desde una URL pública, generada dinámicamente, hasta el puerto `8501` de la máquina virtual de Colab.

4.  **Acceso a la Aplicación**:
    * Finalmente, el script imprime la URL pública. Al hacer clic en dicho enlace, se abrirá el simulador en una nueva pestaña del navegador para que el usuario pueda interactuar con él.

In [6]:
from pyngrok import ngrok

# Pega aquí tu token de autenticación de ngrok
# Puedes obtenerlo gratis desde el dashboard de ngrok: https://dashboard.ngrok.com/get-started/your-authtoken
NGROK_AUTHTOKEN = "2zR6JxbL3xrSc0H4ccPUVnSbAsN_6S1aFw8ckJGrFcVr99J76"
ngrok.set_auth_token(NGROK_AUTHTOKEN)

# Lanza la app de Streamlit en segundo plano en el puerto 8501
!nohup streamlit run streamlit_app.py --server.port 8501 &

# Crea el túnel público con pyngrok
public_url = ngrok.connect(8501)
print("¡Tu dashboard está en vivo! 🚀")
print("URL Pública:", public_url)

nohup: appending output to 'nohup.out'
¡Tu dashboard está en vivo! 🚀
URL Pública: NgrokTunnel: "https://4f424a900943.ngrok-free.app" -> "http://localhost:8501"
